In [1]:
!pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 78.4 MB/s eta 0:00:00
  Attempting uninstall: autograd
    Found existing installation: autograd 1.9.1
    Uninstalling autograd-1.9.1:
      Successfully uninstalled autograd-1.9.1


In [2]:
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qml
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ==========================================================
# 1. VERİ HAZIRLIĞI VE ÖRNEKLEME PROTOKOLÜ (HAKEM #2 YORUM #2)
# ==========================================================
print("--- [KOD 7] KLASİK AE VE KUANTUM QAE (5-FOLD CV) BAŞLATILIYOR ---")
df = pd.read_csv("creditcard.csv")

# Anomali tespiti için 400 Normal + 100 Fraud örneği
normal_df = df[df["Class"] == 0].sample(n=400, random_state=RANDOM_SEED)
fraud_df = df[df["Class"] == 1].sample(n=100, random_state=RANDOM_SEED)

df_all = pd.concat([normal_df, fraud_df]).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

cols_to_drop = [col for col in ["Time", "Class"] if col in df_all.columns]
X_raw = df_all.drop(columns=cols_to_drop).values
y_raw = df_all["Class"].values

print(f"Toplam Veri: {len(X_raw)} (Normal: {np.sum(y_raw == 0)}, Fraud: {np.sum(y_raw == 1)})\n")

n_qubits = 4

# ==========================================================
# 2. MODEL TANIMLARI (KLASİK AE VE KUANTUM QAE)
# ==========================================================
class ClassicalAutoencoder(nn.Module):
    def __init__(self, deep=False):
        super().__init__()
        if not deep:
            # 4 -> 2 -> 4 Sığ Mimari
            self.net = nn.Sequential(
                nn.Linear(n_qubits, 2),
                nn.ReLU(),
                nn.Linear(2, n_qubits)
            )
        else:
            # 4 -> 8 -> 2 -> 8 -> 4 Derin Mimari
            self.net = nn.Sequential(
                nn.Linear(n_qubits, 8),
                nn.ReLU(),
                nn.Linear(8, 2),
                nn.ReLU(),
                nn.Linear(2, 8),
                nn.ReLU(),
                nn.Linear(8, n_qubits)
            )

    def forward(self, x):
        return self.net(x)

    def get_reconstruction_error(self, x):
        with torch.no_grad():
            reconstructed = self.forward(x)
            errors = torch.mean((x - reconstructed) ** 2, dim=1)
        return errors.numpy()

def create_qae_model(n_layers=2):
    dev_qae = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev_qae, interface="torch", diff_method="backprop")
    def qae_qnode(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(n_qubits))
        qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
        # 2. ve 3. kübitler çöp kübit (Trash qubits) olarak Pauli-Z gözlenebilirleriyle ölçülür
        return [qml.expval(qml.PauliZ(2)), qml.expval(qml.PauliZ(3))]

    weight_shapes = {"weights": (n_layers, n_qubits, 3)}

    class QuantumAutoencoder(nn.Module):
        def __init__(self):
            super().__init__()
            self.qlayer = qml.qnn.TorchLayer(qae_qnode, weight_shapes)

        def forward(self, x):
            return self.qlayer(x)

        def get_reconstruction_error(self, x):
            with torch.no_grad():
                z_vals = self.forward(x)
                # <Z>=1 -> |0> temel durumu (hata = 0). <Z> azaldıkça anomali hatası artar.
                # Error = 1 - ((<Z2> + <Z3> + 2) / 4)
                avg_z = (z_vals[:, 0] + z_vals[:, 1] + 2.0) / 4.0
                errors = 1.0 - avg_z
            return errors.numpy()

    return QuantumAutoencoder()

# ==========================================================
# 3. 5-FOLD ANOMALİ TESPİTİ CV MOTORU
# ==========================================================
def evaluate_autoencoder_kfold(model_builder_fn, model_name, is_quantum=False, epochs=30, lr=0.01):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

    fold_auc = []
    fold_acc = []
    fold_f1 = []

    print(f"--- [{model_name}] 5-Fold Çapraz Doğrulama Başlatılıyor ({epochs} Epoch) ---")
    start_total_t = time.time()

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_raw, y_raw), 1):
        # Sızıntısız Ön İşleme: Scaler ve PCA yalnızca train üzerinde fit edilir
        X_tr_raw, X_te_raw = X_raw[train_idx], X_raw[test_idx]
        y_tr, y_te = y_raw[train_idx], y_raw[test_idx]

        scaler = StandardScaler()
        pca = PCA(n_components=n_qubits, random_state=RANDOM_SEED)
        minmax = MinMaxScaler(feature_range=(-np.pi, np.pi))

        X_tr_angles = minmax.fit_transform(pca.fit_transform(scaler.fit_transform(X_tr_raw)))
        X_te_angles = minmax.transform(pca.transform(scaler.transform(X_te_raw)))

        # YARI-DENETİMLİ EĞİTİM: Yalnızca NORMAL (Class == 0) işlemlerle eğitilir!
        normal_mask = (y_tr == 0)
        X_train_normal = X_tr_angles[normal_mask]

        train_tensor = torch.tensor(X_train_normal, dtype=torch.float32)
        test_tensor = torch.tensor(X_te_angles, dtype=torch.float32)
        train_loader = DataLoader(train_tensor, batch_size=32, shuffle=True)

        model = model_builder_fn()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        # Eğitim
        for epoch in range(epochs):
            model.train()
            for bx in train_loader:
                optimizer.zero_grad()
                if not is_quantum:
                    out = model(bx)
                    loss = nn.functional.mse_loss(out, bx)
                else:
                    z_vals = model(bx)
                    # Çöp kübitlerin |0> durumuna (<Z>=1) kilitlenmesi hedeflenir
                    loss = torch.mean(1.0 - (z_vals[:, 0] + z_vals[:, 1] + 2.0) / 4.0)
                loss.backward()
                optimizer.step()

        # Doğrulama / Test Aşaması
        model.eval()
        train_errors = model.get_reconstruction_error(train_tensor)
        test_errors = model.get_reconstruction_error(test_tensor)

        # EŞİK BELİRLEME (Test etiketlerine bakmadan, train normal verisinin %95 persentili)
        threshold = np.percentile(train_errors, 95)
        test_preds = (test_errors > threshold).astype(int)

        auc = roc_auc_score(y_te, test_errors)
        acc = accuracy_score(y_te, test_preds)
        f1 = f1_score(y_te, test_preds, pos_label=1, zero_division=0)

        fold_auc.append(auc)
        fold_acc.append(acc)
        fold_f1.append(f1)

        print(f"   Fold {fold}/5 -> ROC-AUC: %{auc*100:.2f} | Doğruluk: %{acc*100:.2f} | F1: %{f1*100:.2f}")

    elapsed_time = time.time() - start_total_t
    print(f"[{model_name}] Tamamlandı (Süre: {elapsed_time:.1f} sn)\n")

    return {
        "auc_mean": np.mean(fold_auc) * 100, "auc_std": np.std(fold_auc) * 100,
        "acc_mean": np.mean(fold_acc) * 100, "acc_std": np.std(fold_acc) * 100,
        "f1_mean": np.mean(fold_f1) * 100, "f1_std": np.std(fold_f1) * 100,
        "time": elapsed_time
    }

# ==========================================================
# 4. 5-FOLD MODELLERİN KOŞUMU
# ==========================================================
ae_results = {}

# Klasik AE (Sığ 4-2-4)
ae_results["Klasik AE (Sığ 4-2-4, 30 Ep)"] = evaluate_autoencoder_kfold(
    lambda: ClassicalAutoencoder(deep=False),
    "Klasik AE (Sığ)", is_quantum=False, epochs=30, lr=0.005
)

# Klasik AE (Derin 4-8-2-8-4)
ae_results["Klasik AE (Derin 4-8-2-8-4, 30 Ep)"] = evaluate_autoencoder_kfold(
    lambda: ClassicalAutoencoder(deep=True),
    "Klasik AE (Derin)", is_quantum=False, epochs=30, lr=0.005
)

# Kuantum QAE (2 Katmanlı, 2 Çöp Kübit)
ae_results["Kuantum QAE (2 Katman, 30 Ep)"] = evaluate_autoencoder_kfold(
    lambda: create_qae_model(n_layers=2),
    "Kuantum QAE (2 Katman)", is_quantum=True, epochs=30, lr=0.01
)

# ==========================================================
# 5. HAKEM #2 YORUM #4 VE #10 İÇİN NİHAİ RAPOR TABLOSU
# ==========================================================
print("=" * 105)
print("HAKEM RAPORU: AUTOENCODER ANOMALİ TESPİTİ 5-FOLD STRATIFIED CV SONUÇLARI (Mean ± SD)")
print("=" * 105)
print(f"{'Model Mimarisi':<35} | {'ROC-AUC (Mean ± SD)':<22} | {'Doğruluk (Mean ± SD)':<22} | {'F1-Skoru (Mean ± SD)'}")
print("-" * 105)
for name, res in ae_results.items():
    auc_str = f"%{res['auc_mean']:.2f} ± {res['auc_std']:.2f}"
    acc_str = f"%{res['acc_mean']:.2f} ± {res['acc_std']:.2f}"
    f1_str = f"%{res['f1_mean']:.2f} ± {res['f1_std']:.2f}"
    print(f"{name:<35} | {auc_str:<22} | {acc_str:<22} | {f1_str}")
print("=" * 105)

--- [KOD 7] KLASİK AE VE KUANTUM QAE (5-FOLD CV) BAŞLATILIYOR ---
Toplam Veri: 500 (Normal: 400, Fraud: 100)

--- [Klasik AE (Sığ)] 5-Fold Çapraz Doğrulama Başlatılıyor (30 Epoch) ---
   Fold 1/5 -> ROC-AUC: %98.50 | Doğruluk: %96.00 | F1: %90.00
   Fold 2/5 -> ROC-AUC: %98.94 | Doğruluk: %96.00 | F1: %90.48
   Fold 3/5 -> ROC-AUC: %40.94 | Doğruluk: %82.00 | F1: %35.71
   Fold 4/5 -> ROC-AUC: %87.50 | Doğruluk: %93.00 | F1: %81.08
   Fold 5/5 -> ROC-AUC: %41.88 | Doğruluk: %83.00 | F1: %48.48
[Klasik AE (Sığ)] Tamamlandı (Süre: 7.5 sn)

--- [Klasik AE (Derin)] 5-Fold Çapraz Doğrulama Başlatılıyor (30 Epoch) ---
   Fold 1/5 -> ROC-AUC: %99.12 | Doğruluk: %97.00 | F1: %92.68
   Fold 2/5 -> ROC-AUC: %98.69 | Doğruluk: %96.00 | F1: %90.48
   Fold 3/5 -> ROC-AUC: %85.62 | Doğruluk: %91.00 | F1: %76.92
   Fold 4/5 -> ROC-AUC: %87.50 | Doğruluk: %93.00 | F1: %81.08
   Fold 5/5 -> ROC-AUC: %91.31 | Doğruluk: %90.00 | F1: %77.27
[Klasik AE (Derin)] Tamamlandı (Süre: 1.6 sn)

--- [Kuantum QAE (

In [3]:
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qml
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ==========================================================
# 1. VERİ HAZIRLIĞI (HAKEM #2 YORUM #2)
# ==========================================================
print("--- [KOD 7] SABİT KLASİK AE VE SABİT QAE (5-FOLD CV) BAŞLATILIYOR ---")
df = pd.read_csv("creditcard.csv")

# 400 Normal + 100 Fraud = 500 Toplam Örneklem
normal_df = df[df["Class"] == 0].sample(n=400, random_state=RANDOM_SEED)
fraud_df = df[df["Class"] == 1].sample(n=100, random_state=RANDOM_SEED)

df_all = (
    pd.concat([normal_df, fraud_df])
    .sample(frac=1, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)

cols_to_drop = [col for col in ["Time", "Class"] if col in df_all.columns]
X_raw = df_all.drop(columns=cols_to_drop).values
y_raw = df_all["Class"].values

print(f"Toplam Örneklem: {len(X_raw)} (Normal: {np.sum(y_raw == 0)}, Fraud: {np.sum(y_raw == 1)})\n")

n_qubits = 4

# ==========================================================
# 2. SABİT MODELLERİN MİMARİ TANIMLARI
# ==========================================================
class ClassicalAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        # Sabit 4 -> 8 -> 2 -> 8 -> 4 Mimarisi
        self.net = nn.Sequential(
            nn.Linear(n_qubits, 8),
            nn.ReLU(),
            nn.Linear(8, 2),
            nn.ReLU(),
            nn.Linear(2, 8),
            nn.ReLU(),
            nn.Linear(8, n_qubits)
        )

    def forward(self, x):
        return self.net(x)

    def get_reconstruction_error(self, x):
        with torch.no_grad():
            reconstructed = self.forward(x)
            errors = torch.mean((x - reconstructed) ** 2, dim=1)
        return errors.numpy()

# Sabit 2 Katmanlı QAE Devresi
fixed_layers = 2
dev_qae = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev_qae, interface="torch", diff_method="backprop")
def qae_qnode(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    # 2. ve 3. kübitler çöp kübit (Trash qubits) olarak ölçülür
    return [qml.expval(qml.PauliZ(2)), qml.expval(qml.PauliZ(3))]

weight_shapes = {"weights": (fixed_layers, n_qubits, 3)}

class QuantumAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.qlayer = qml.qnn.TorchLayer(qae_qnode, weight_shapes)

    def forward(self, x):
        return self.qlayer(x)

    def get_reconstruction_error(self, x):
        with torch.no_grad():
            z_vals = self.forward(x)
            # <Z>=1 -> |0> temel durumu (hata = 0). Anomali arttıkça Z düşer.
            avg_z = (z_vals[:, 0] + z_vals[:, 1] + 2.0) / 4.0
            errors = 1.0 - avg_z
        return errors.numpy()

# ==========================================================
# 3. 5-FOLD STRATIFIED ANOMALİ CV MOTORU
# ==========================================================
def run_autoencoder_cv(model_cls, model_name, is_quantum=False, epochs=30, lr=0.01):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    fold_auc, fold_acc, fold_f1 = [], [], []

    print(f"[{model_name}] 5-Fold CV Başlatılıyor ({epochs} Epoch)...")
    start_total_t = time.time()

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_raw, y_raw), 1):
        # 1. Veri Sızıntısını Önleme: Fit işlemleri YALNIZCA train setine
        X_tr_raw, X_te_raw = X_raw[train_idx], X_raw[test_idx]
        y_tr, y_te = y_raw[train_idx], y_raw[test_idx]

        scaler = StandardScaler()
        pca = PCA(n_components=n_qubits, random_state=RANDOM_SEED)
        minmax = MinMaxScaler(feature_range=(-np.pi, np.pi))

        X_tr_angles = minmax.fit_transform(pca.fit_transform(scaler.fit_transform(X_tr_raw)))
        X_te_angles = minmax.transform(pca.transform(scaler.transform(X_te_raw)))

        # 2. YARI-DENETİMLİ EĞİTİM: Yalnızca NORMAL (Class == 0) verilerle eğitilir
        normal_mask = (y_tr == 0)
        X_train_normal = X_tr_angles[normal_mask]

        train_tensor = torch.tensor(X_train_normal, dtype=torch.float32)
        test_tensor = torch.tensor(X_te_angles, dtype=torch.float32)
        train_loader = DataLoader(train_tensor, batch_size=32, shuffle=True)

        model = model_cls()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        # Eğitim
        for epoch in range(epochs):
            model.train()
            for bx in train_loader:
                optimizer.zero_grad()
                if not is_quantum:
                    out = model(bx)
                    loss = nn.functional.mse_loss(out, bx)
                else:
                    z_vals = model(bx)
                    loss = torch.mean(1.0 - (z_vals[:, 0] + z_vals[:, 1] + 2.0) / 4.0)
                loss.backward()
                optimizer.step()

        # 3. SIZINTISIZ EŞİKLEME (Hakem #2 Yorum #10):
        # Eşik test verisine bakılmadan, eğitimdeki normal verilerin 95. persentili ile belirlenir
        model.eval()
        train_errors = model.get_reconstruction_error(train_tensor)
        test_errors = model.get_reconstruction_error(test_tensor)

        threshold = np.percentile(train_errors, 95)
        test_preds = (test_errors > threshold).astype(int)

        auc = roc_auc_score(y_te, test_errors)
        acc = accuracy_score(y_te, test_preds)
        f1 = f1_score(y_te, test_preds, pos_label=1, zero_division=0)

        fold_auc.append(auc)
        fold_acc.append(acc)
        fold_f1.append(f1)

    elapsed_time = time.time() - start_total_t
    print(f"   -> {model_name} Tamamlandı (Süre: {elapsed_time:.1f} sn | AUC: %{np.mean(fold_auc)*100:.2f} ± {np.std(fold_auc)*100:.2f})\n")

    return {
        "auc_mean": np.mean(fold_auc) * 100, "auc_std": np.std(fold_auc) * 100,
        "acc_mean": np.mean(fold_acc) * 100, "acc_std": np.std(fold_acc) * 100,
        "f1_mean": np.mean(fold_f1) * 100, "f1_std": np.std(fold_f1) * 100,
        "time": elapsed_time
    }

# ==========================================================
# 4. MODELLERİN FARKLI EPOCHLARDA 5-FOLD YARIŞTIRILMASI
# ==========================================================
print("=== 5-FOLD ÇAPRAZ DOĞRULAMA KOŞUMLARI BAŞLIYOR ===\n")
cv_results = {}

# --- SABİT KUANTUM QAE (2 KATMAN) ---
cv_results["QAE (2 Katman, 20 Ep)"] = run_autoencoder_cv(QuantumAutoencoder, "QAE (20 Ep)", is_quantum=True, epochs=20, lr=0.02)
cv_results["QAE (2 Katman, 50 Ep)"] = run_autoencoder_cv(QuantumAutoencoder, "QAE (50 Ep)", is_quantum=True, epochs=50, lr=0.02)
cv_results["QAE (2 Katman, 100 Ep)"] = run_autoencoder_cv(QuantumAutoencoder, "QAE (100 Ep)", is_quantum=True, epochs=100, lr=0.02)

# --- SABİT KLASİK AE (4-8-2-8-4) ---
cv_results["Klasik AE (Sabit, 20 Ep)"] = run_autoencoder_cv(ClassicalAutoencoder, "Klasik AE (20 Ep)", is_quantum=False, epochs=20, lr=0.005)
cv_results["Klasik AE (Sabit, 50 Ep)"] = run_autoencoder_cv(ClassicalAutoencoder, "Klasik AE (50 Ep)", is_quantum=False, epochs=50, lr=0.005)
cv_results["Klasik AE (Sabit, 100 Ep)"] = run_autoencoder_cv(ClassicalAutoencoder, "Klasik AE (100 Ep)", is_quantum=False, epochs=100, lr=0.005)

# ==========================================================
# 5. HAKEM #2 YORUM #4 VE #10 İÇİN NİHAİ RAPOR TABLOSU
# ==========================================================
print("\n" + "=" * 105)
print("HAKEM RAPORU: 5-FOLD STRATIFIED CV AUTOENCODER SONUÇLARI (Mean ± SD)")
print("=" * 105)
print(f"{'Model / Epoch':<30} | {'ROC-AUC (Mean ± SD)':<22} | {'Doğruluk (Mean ± SD)':<22} | {'F1-Skoru (Mean ± SD)'}")
print("-" * 105)
for name, res in cv_results.items():
    auc_str = f"%{res['auc_mean']:.2f} ± {res['auc_std']:.2f}"
    acc_str = f"%{res['acc_mean']:.2f} ± {res['acc_std']:.2f}"
    f1_str = f"%{res['f1_mean']:.2f} ± {res['f1_std']:.2f}"
    print(f"{name:<30} | {auc_str:<22} | {acc_str:<22} | {f1_str}")
print("=" * 105)

--- [KOD 7] SABİT KLASİK AE VE SABİT QAE (5-FOLD CV) BAŞLATILIYOR ---
Toplam Örneklem: 500 (Normal: 400, Fraud: 100)

=== 5-FOLD ÇAPRAZ DOĞRULAMA KOŞUMLARI BAŞLIYOR ===

[QAE (20 Ep)] 5-Fold CV Başlatılıyor (20 Epoch)...
   -> QAE (20 Ep) Tamamlandı (Süre: 19.5 sn | AUC: %91.89 ± 5.35)

[QAE (50 Ep)] 5-Fold CV Başlatılıyor (50 Epoch)...
   -> QAE (50 Ep) Tamamlandı (Süre: 43.5 sn | AUC: %91.72 ± 5.39)

[QAE (100 Ep)] 5-Fold CV Başlatılıyor (100 Epoch)...
   -> QAE (100 Ep) Tamamlandı (Süre: 90.5 sn | AUC: %92.42 ± 5.65)

[Klasik AE (20 Ep)] 5-Fold CV Başlatılıyor (20 Epoch)...
   -> Klasik AE (20 Ep) Tamamlandı (Süre: 1.2 sn | AUC: %92.42 ± 5.52)

[Klasik AE (50 Ep)] 5-Fold CV Başlatılıyor (50 Epoch)...
   -> Klasik AE (50 Ep) Tamamlandı (Süre: 2.7 sn | AUC: %92.50 ± 5.41)

[Klasik AE (100 Ep)] 5-Fold CV Başlatılıyor (100 Epoch)...
   -> Klasik AE (100 Ep) Tamamlandı (Süre: 4.6 sn | AUC: %91.38 ± 5.36)


HAKEM RAPORU: 5-FOLD STRATIFIED CV AUTOENCODER SONUÇLARI (Mean ± SD)
Model / Epoch